In [1]:
import numpy as np

In [2]:

phrase1 = ["je", "mange", "une", "pomme", "rouge"]
phrase2 = ["je", "bois", "un", "jus", "de", "pomme"]
phrase3 = ["une", "pomme", "est", "un", "fruit"]
phrase4 = ["la", "banane", "est", "un", "fruit", "jaune"]
phrase5 = ["je", "mange", "une", "banane"]

In [3]:
corpus = [phrase1, phrase2, phrase3, phrase4, phrase5]
vocab = ["banane", "bois", "de", "est", "fruit", "jaune", "je", "jus", "la", "mange", "pomme", "rouge", "un", "une"]
w2i = {w:i for i,w in enumerate(vocab)}
V = len(vocab)

In [4]:
X = np.zeros((V, V))
for sent in corpus:
    ids = [w2i[w] for w in sent]
    for i, c in enumerate(ids):
        for j in range(max(0,i-2), min(len(ids), i+3)):
            if i != j:
                X[c, ids[j]] += 1/abs(i-j)

In [5]:
np.random.seed(42)
d = 8
W, Wt = np.random.randn(V,d)*0.5, np.random.randn(V,d)*0.5
b, bt = np.zeros(V), np.zeros(V)

for epoch in range(400):
    for i in range(V):
        for j in range(V):
            if X[i,j]==0: continue
            f = (X[i,j]/10)**0.75  # fonction de pondération
            # Objectif GloVe: w_i^T wj + b_i + bj = log(Xij)
            diff = np.dot(W[i], Wt[j]) + b[i] + bt[j] - np.log(X[i,j])
            g = f * diff
            Wi_old = W[i].copy()
            W[i] -= 0.05 * g * Wt[j]
            Wt[j] -= 0.05 * g * Wi_old
            b[i] -= 0.05 * g
            bt[j] -= 0.05 * g

E = (W + Wt) / 2  # vecteurs finaux

In [6]:
vec_pomme = E[w2i["pomme"]]
print("Vecteur pomme =", vec_pomme)

Vecteur pomme = [-0.03394749 -0.39722471  0.43819683  0.08028184  0.05570621 -0.12150039
  0.32981955 -0.01808795]


In [7]:
def vec_phrase(words):
    return np.mean([E[w2i[w]] for w in words], axis=0)

def cos_sim(a,b):
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b)))

v1 = vec_phrase(phrase1)
v2 = vec_phrase(phrase2)
v4 = vec_phrase(phrase4)

print("Cos(P2,P1) =", cos_sim(v2, v1))
print("Cos(P2,P4) =", cos_sim(v2, v4))

Cos(P2,P1) = -0.20008161728689264
Cos(P2,P4) = 0.44037965374269444
